In [1]:
import numpy as np
import pandas as pd
import psycopg
from sqlalchemy import create_engine
import dotenv
import os

dotenv.load_dotenv()

True

In [2]:
POSTGRES_PASSWORD = os.getenv('POSTGRES_PASSWORD')

In [3]:
games = pd.read_csv('games.csv')
teams = pd.read_csv('teams.csv')
players = pd.read_csv('players.csv')
team_game = pd.read_csv('team_game.csv')
player_game = pd.read_csv('player_game.csv')

# Step1: Connect to the Postgres (on docker) serer, create an empty NBA database



In [4]:
dbserver = psycopg.connect(
    user = 'postgres',
    password = POSTGRES_PASSWORD,
    host = 'localhost',
    port = '5432')
dbserver.autocommit = True

In [5]:
cursor = dbserver.cursor()

In [6]:
cursor.execute('CREATE DATABASE nba')

<psycopg.Cursor [COMMAND_OK] [IDLE] (host=localhost database=postgres) at 0x2020447c050>

In [7]:
try:
    cursor.execute('CREATE DATABASE nba')
except:
    cursor.execute('DROP DATABASE nba')
    cursor.execute('CREATE DATABASE nba')

# Step 2: Add our data to this database



In [8]:
dbms = 'postgresql'
package = 'psycopg'
user = 'postgres'
password = 'POSTGRES_PASSWORD' 
host = 'localhost'
port = '5432'
db = 'nba'
engine = create_engine(f'{dbms}+{package}://{user}:{password}@{host}:{port}/{db}')

In [ ]:
games.to_sql('games', con=engine, index=False, chunksize=1000, if_exists='replace')
teams.to_sql('teams', con=engine, index=False, chunksize=1000, if_exists='replace')
players.to_sql('players', con=engine, index=False, chunksize=1000, if_exists='replace')
team_game.to_sql('team_game', con=engine, index=False, chunksize=1000, if_exists='replace')
player_game.to_sql('player_game', con=engine, index=False, chunksize=1000, if_exists='replace')

OperationalError: (psycopg.OperationalError) connection failed: connection to server at "127.0.0.1", port 5432 failed: could not initiate GSSAPI security context: No credentials were supplied, or the credentials were unavailable or inaccessible: Credential cache is empty
connection to server at "127.0.0.1", port 5432 failed: FATAL:  password authentication failed for user "postgres"
Multiple connection attempts failed. All failures were:
- host: 'localhost', port: 5432, hostaddr: '::1': connection failed: connection to server at "::1", port 5432 failed: could not initiate GSSAPI security context: No credentials were supplied, or the credentials were unavailable or inaccessible: Credential cache is empty
connection to server at "::1", port 5432 failed: FATAL:  password authentication failed for user "postgres"
- host: 'localhost', port: 5432, hostaddr: '127.0.0.1': connection failed: connection to server at "127.0.0.1", port 5432 failed: could not initiate GSSAPI security context: No cr

: 

# Step 3: SQL it like crazy

In [ ]:
myquery = '''
SELECT *
FROM games
'''

myquery2 = '''
SELECT pg.player_id, p.player_name,
    SUM(pg.pts + pg.fg3m - pg.fga + 2*pg.fgm - pg.fta + pg.ftm + pg.oreb + pg.dreb + 2*pg.ast + 4*pg.stl + 4*pg.blk - 2*pg.tov) AS mvp_points
FROM player_game pg
INNER JOIN games g
    ON pg.game_id = g.game_id
INNER JOIN players p
    ON pg.player_id = p.player_id
WHERE g.season_type = 'Regular Season'
'''

pd.read_sql_query(myquery, con=engine)

Who should've been the MVP last season?